# VE SUNAVAL — Superintendencia Nacional de Valores (Venezuela)

Source: Jira DECD-4399. 9 PHP-backed list pages on `https://www.sunaval.gob.ve/`.

**Site behavior** (per user observation):
- Each list page renders an empty `<table>` then fills it via XHR (PHP backend).
- Each row has an action **button** that, when clicked, reveals the entity address.
- The address info must be captured and kept attached to that row.

**Strategy:** Selenium drives the page so the XHR completes and the row-buttons can be clicked. BeautifulSoup parses the populated table; for each row we click the detail button and harvest the address from the resulting panel/modal.

⚠️ **TODO before running** — fill in the two placeholders flagged with `# TODO:` in the main loop (the XHR/wait selector for the table, and the address-button selector + the container the address appears in). The structure below is ready; only the two selectors are unknown until the page is inspected in DevTools.

In [ ]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException
from time import sleep
import os
import random
import requests

In [ ]:
#------------------------------------------------ Begin_fileName ----------------------------------------
regulatorName = 'VE SUNAVAL'

print(f"Running {regulatorName} Web Scraping Tool v.1.0")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

In [ ]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
chromeOptions = webdriver.ChromeOptions()
prefs = {
    "plugins.always_open_pdf_externally": True,
    "download.prompt_for_download": False,
    "download.default_directory": tempfolder,
    'profile.default_content_setting_values.automatic_downloads': 1
}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

In [ ]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict = {
    'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [],
    'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 'InternalID_2_type': [],
    'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [],
    'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [],
    'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [],
    'CancellationDate': [], 'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [],
    'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [],
    'Name - Mother Company': [], 'Address_1 - Mother company': [], 'Address_2 -  Mother company': [],
    'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
    'Phone - Mother company': [], 'Check': []
}

processdate = now.strftime('%Y-%m-%d')

regdict = {
    regulatorName + ' 1': 'https://www.sunaval.gob.ve/asesores-de-inversion-jur/',
    regulatorName + ' 2': 'https://www.sunaval.gob.ve/sociedades-de-corretaje-de-inversion/',
    regulatorName + ' 3': 'https://www.sunaval.gob.ve/casas-de-bolsas-de-productos-agricolas/',
    regulatorName + ' 4': 'https://www.sunaval.gob.ve/fondos-mutuales/',
    regulatorName + ' 5': 'https://www.sunaval.gob.ve/sociedades-administradoras/',
    regulatorName + ' 6': 'https://www.sunaval.gob.ve/sociedades-calificadoras-de-riesgos/',
    regulatorName + ' 7': 'https://www.sunaval.gob.ve/agente-de-traspasos/',
    regulatorName + ' 8': 'https://www.sunaval.gob.ve/sociedad-titularizadora/',
    regulatorName + ' 9': 'https://www.sunaval.gob.ve/otro-ente/',
}

Typology = {
    regulatorName + ' 1': 'List of Investment Advisors (Legal Persons)',
    regulatorName + ' 2': 'List of Investment Brokerage Companies',
    regulatorName + ' 3': 'List of Agricultural Products Brokerage Houses',
    regulatorName + ' 4': 'List of Mutual Funds',
    regulatorName + ' 5': 'List of Management Companies',
    regulatorName + ' 6': 'List of Risk Rating Companies',
    regulatorName + ' 7': 'List of Transfer Agents',
    regulatorName + ' 8': 'List of Securitization Companies',
    regulatorName + ' 9': 'List of Other Entities',
}

In [ ]:
#------------------------------------------------ Begin_Helper ----------------------------------------
def pad_dict(sqldict):
    """Make every list in sqldict the same length as ListProcessDate."""
    maxlen = len(sqldict['ListProcessDate'])
    for key in sqldict:
        if len(sqldict[key]) < maxlen:
            sqldict[key] += [''] * (maxlen - len(sqldict[key]))
    return sqldict


def parse_address_block(text):
    """Split a free-form Spanish address into Address_1 / City / Phone if possible.
    SUNAVAL detail panels typically expose: Direccion, Telefono, RIF, etc.
    Anything we can't classify stays in Address_1."""
    out = {'Address_1': '', 'City': '', 'Phone': '', 'Email': '', 'Website': ''}
    if not text:
        return out
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    leftover = []
    for line in lines:
        low = line.lower()
        if low.startswith(('direcc', 'dirección', 'direccion')):
            out['Address_1'] = line.split(':', 1)[-1].strip()
        elif low.startswith(('tel', 'telef', 'teléf')):
            out['Phone'] = line.split(':', 1)[-1].strip()
        elif low.startswith(('correo', 'email', 'e-mail')):
            out['Email'] = line.split(':', 1)[-1].strip()
        elif low.startswith(('web', 'sitio', 'pagina', 'página')):
            out['Website'] = line.split(':', 1)[-1].strip()
        elif low.startswith(('ciudad', 'estado', 'municipio')):
            out['City'] = line.split(':', 1)[-1].strip()
        else:
            leftover.append(line)
    if not out['Address_1'] and leftover:
        out['Address_1'] = ' | '.join(leftover)
    return out

### XHR / detail-button reconnaissance — fill these in once

Open one of the list pages in Chrome DevTools → **Network** → **Fetch/XHR**. Refresh, and note:

1. **Table-row XHR**  — URL + method (likely `POST` to `admin-ajax.php` or a `/wp-json/` route). If JSON, the scraper can hit it directly with `requests` and skip Selenium entirely.
2. **Address button** — CSS selector for the button inside each row (e.g. `td.ficha a`, `button.ver-direccion`) **and** the selector for the container that appears on click (modal, popover, expanded `<tr>`).

Set them below. Until they're set the loop will still scrape Name/RIF/Description/Estado from the populated table; only the address column will stay empty.

In [ ]:
TABLE_READY_SELECTOR = 'table tbody tr'

DETAIL_BUTTON_SELECTOR = 'td:last-child a, td:last-child button'
DETAIL_PANEL_SELECTOR = '.modal.show .modal-body, .ficha-detalle'
DETAIL_CLOSE_SELECTOR = '.modal.show .btn-close, .modal.show .close'

XHR_ENDPOINT = None
XHR_PAYLOADS = {}

In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------
for reg, url in regdict.items():
    list_name = Typology.get(reg, 'Unknown List')
    print(f'Working with {reg} - {list_name}')

    try:
        driver.get(url)
    except Exception as e:
        print(f'  [WARN] navigation failed: {e}')
        continue

    try:
        WebDriverWait(driver, 25).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, TABLE_READY_SELECTOR))
        )
    except TimeoutException:
        print(f'  [WARN] table never populated for {reg} (XHR timeout). Skipping.')
        continue
    sleep(random.uniform(2, 4))

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    table = soup.find('table')
    headers = [th.get_text(strip=True) for th in table.find_all('th')] if table else []
    print(f'  [INFO] headers: {headers}')

    row_elements = driver.find_elements(By.CSS_SELECTOR, 'table tbody tr')
    print(f'  [INFO] found {len(row_elements)} rows on {reg}')

    for idx, row_el in enumerate(row_elements):
        try:
            cells = row_el.find_elements(By.TAG_NAME, 'td')
            if not cells:
                continue

            cell_text = [c.text.strip() for c in cells]
            rif = cell_text[0] if len(cell_text) > 0 else ''
            descripcion = cell_text[1] if len(cell_text) > 1 else ''
            estado = cell_text[2] if len(cell_text) > 2 else ''

            address_info = {'Address_1': '', 'City': '', 'Phone': '', 'Email': '', 'Website': ''}
            try:
                btn = row_el.find_element(By.CSS_SELECTOR, DETAIL_BUTTON_SELECTOR)
                driver.execute_script('arguments[0].scrollIntoView({block: "center"});', btn)
                sleep(0.3)
                try:
                    btn.click()
                except ElementClickInterceptedException:
                    driver.execute_script('arguments[0].click();', btn)

                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, DETAIL_PANEL_SELECTOR))
                )
                sleep(random.uniform(0.8, 1.5))
                panel = driver.find_element(By.CSS_SELECTOR, DETAIL_PANEL_SELECTOR)
                address_info = parse_address_block(panel.text)

                try:
                    close_btn = driver.find_element(By.CSS_SELECTOR, DETAIL_CLOSE_SELECTOR)
                    close_btn.click()
                    sleep(0.4)
                except NoSuchElementException:
                    driver.execute_script('document.body.click();')
                    sleep(0.4)
            except (NoSuchElementException, TimeoutException):
                pass

            sqldict['Name'].append(descripcion or rif)
            sqldict['InternalID_1'].append(rif)
            sqldict['InternalID_1_type'].append('RIF' if rif else '')
            sqldict['RegulationType'].append(estado or 'Regulated')
            sqldict['Address_1'].append(address_info['Address_1'])
            sqldict['City'].append(address_info['City'])
            sqldict['Phone'].append(address_info['Phone'])
            sqldict['Email'].append(address_info['Email'])
            sqldict['Website'].append(address_info['Website'])

            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append('VE')
            sqldict['RegCode'].append('SUNAVAL')
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(list_name)
            sqldict['ListLanguage'].append('Spanish')
            sqldict['Cntry'].append('Venezuela')

            sqldict = pad_dict(sqldict)
        except Exception as e:
            print(f'  [WARN] row {idx} failed: {e}')
            continue

    print(f'  [INFO] Completed {reg}')
    sleep(random.uniform(2, 5))

In [ ]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(filename, 'SQL Ready', index=False)
driver.quit()
sleep(3)
print(f"[INFO] : Excel file '{filename}' saved successfully — {len(df)} rows")